# D39 - 내 사진으로 학습하고 best.pt 받기

위에서부터 셀을 하나씩 실행합니다. 셀을 만들거나 코드를 붙여넣지 않습니다.
각 셀 왼쪽의 재생(▶) 단추를 한 번만 누르고, 아래 결과가 나오면 다음 셀로 갑니다.

- 시작 전: 메뉴 `런타임` → `런타임 유형 변경`에서 `T4 GPU`를 고르고 `저장`합니다. 무료 선택지만 씁니다. `Colab Pro+ 업그레이드` 안내는 누르지 않습니다.
- 카드 입력 화면이 나오면 멈추고 손을 듭니다.
- `런타임 연결이 끊어짐`이 나오면 다시 연결합니다. 새 컴퓨터가 되므로 셀 1부터 다시 실행합니다.

## 셀 1 - GPU 확인
`!nvidia-smi`는 연결된 GPU 상태를 보여 줍니다. `Tesla T4`가 보이면 됩니다.

In [ ]:
!nvidia-smi

## 셀 2 - 학습 도구 설치
`ultralytics`는 YOLO를 학습하는 도구입니다. 설치가 끝나면 `ultralytics==` 줄이 보입니다.

In [ ]:
!uv pip install ultralytics

## 셀 3 - ZIP 올리기
실행하면 `파일 선택` 단추가 나옵니다. 내 PC에서 내가 받은 ZIP(이름은 달라도 됨)을 직접 고릅니다. 취소하면 이 셀을 다시 실행합니다.

In [ ]:
from google.colab import files
uploaded = files.upload()

## 셀 4 - ZIP 풀기
올린 ZIP을 자동으로 찾아 `/content/data`에 풉니다. ZIP 이름에 공백이 있어도 됩니다. ZIP이 하나가 아니면 멈춥니다.

In [ ]:
import glob
zips = glob.glob("/content/*.zip")
if len(zips) != 1:
    raise ValueError(f"ZIP이 {len(zips)}개입니다. 왼쪽 파일 패널에서 하나만 남기세요: {zips}")
zip_path = zips[0]
print("찾은 ZIP:", zip_path)
!unzip -q -o "{zip_path}" -d /content/data

## 셀 5 - data.yaml 경로를 Colab에 맞추기
원본은 보존하고 Colab 위치를 기준으로 한 data_colab.yaml 사본을 만듭니다. 경로와 names 출력이 내 ZIP과 맞는지 확인합니다.


In [ ]:
from pathlib import Path
root = Path("/content/data")
source = root / "data.yaml"
text = source.read_text(encoding="utf-8")
if "train: ../train/images" not in text or "val: ../valid/images" not in text or "test: ../test/images" not in text:
    raise ValueError("data.yaml 경로 형식을 강사와 확인하세요")
text = text.replace("train: ../train/images", "train: train/images")
text = text.replace("val: ../valid/images", "val: valid/images")
text = text.replace("test: ../test/images", "test: test/images")
target = root / "data_colab.yaml"
target.write_text(f"path: {root}\n" + text, encoding="utf-8")
print(target.read_text(encoding="utf-8"))


## 셀 6 - 학습 (한 번만 실행)
맨 위 `EPOCHS`가 학습을 되풀이할 횟수입니다. 강사가 알려 준 값으로 두고 셀을 한 번만 실행합니다. 같은 셀을 다시 누르지 않습니다.

In [ ]:
EPOCHS = 30  # [학생 70장 규모 실측 후 확정] 20~30 권장
!yolo detect train data=/content/data/data_colab.yaml model=yolo26n.pt epochs={EPOCHS} imgsz=640 batch=16 project=/content/runs name=mine

## 셀 7 - 결과 파일 확인
`best.pt`와 `last.pt`가 보이는지 읽습니다.

In [ ]:
!ls -la /content/runs/mine/weights

## 셀 8 - best.pt를 내 PC로 받기
실행하면 브라우저가 파일을 받습니다. 받은 파일은 `d38_photo` 폴더에 `my_best.pt`로 이름을 바꿔 둡니다.

In [ ]:
from google.colab import files
files.download('/content/runs/mine/weights/best.pt')